<a href="https://colab.research.google.com/github/Shajs23224224/FFAIAssistant/blob/main/colab/FFAI_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Free Fire AI - Colab + Drive + Flask + Ngrok

**Arquitectura mejorada:**
- 🧠 Colab = Motor IA (GPU)
- 💾 Drive = Persistencia (modelos, logs)
- 🌐 Flask API = Interfaz REST/WebSocket
- 🔗 Ngrok = URL pública
- 📱 Cliente = APK Android

## Instrucciones:
1. **Runtime → Run all** (Ctrl+F9)
2. Espera la URL de ngrok
3. Copia la URL en ServerConfig.kt del APK
4. ¡Juega!

## Celda 1: Montar Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive montado en /content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado en /content/drive


## Celda 2: Instalar Dependencias

In [10]:
!pip install -q flask flask-socketio flask-cors pyngrok torch torchvision pillow numpy opencv-python-headless
print("✅ Dependencias instaladas")

✅ Dependencias instaladas


## Celda 3: Crear Módulos del Sistema

In [11]:
# Crear estructura de directorios
import os
os.makedirs('/content/ffai_core', exist_ok=True)

# Guardar engine.py
engine_code = '''"""Motor de IA"""

import torch
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
import io
import base64
import time
from typing import Dict, Tuple, List


class TacticalNN(nn.Module):
    def __init__(self, hidden=256, actions=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(128, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, 128), nn.ReLU(),
            nn.Linear(128, actions)
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)


class FFAIEngine:
    ACTIONS = {
        0: ("HOLD", 960, 700, 100),
        1: ("SHOOT", 1150, 750, 100),
        2: ("AIM", 1150, 600, 200),
        3: ("FORWARD", 960, 500, 300),
        4: ("BACK", 960, 900, 300),
        5: ("LEFT", 700, 700, 300),
        6: ("RIGHT", 1220, 700, 300),
        7: ("RELOAD", 1200, 900, 1500),
        8: ("HEAL", 150, 800, 500),
        9: ("JUMP", 960, 700, 100),
    }

    def __init__(self, device="cpu"):
        self.device = torch.device(device)
        self.model = TacticalNN().to(self.device)
        self.model.eval()
        self.frame_count = 0
        self.session_start = time.time()
        print(f"🧠 Engine listo en {device}")

    def decode_image(self, b64):
        try:
            img_bytes = base64.b64decode(b64)
            img = Image.open(io.BytesIO(img_bytes))
            return np.array(img)
        except:
            return np.zeros((135, 240, 3), dtype=np.uint8)

    def analyze_frame(self, img):
        h, w = img.shape[:2]
        health_zone = img[int(h*0.88):int(h*0.96), int(w*0.02):int(w*0.22)]
        ammo_zone = img[int(h*0.88):int(h*0.96), int(w*0.78):int(w*0.98)]
        enemy_zone = img[int(h*0.15):int(h*0.55), int(w*0.35):int(w*0.65)]

        hsv = cv2.cvtColor(health_zone, cv2.COLOR_RGB2HSV)
        green = cv2.inRange(hsv, np.array([35,40,40]), np.array([85,255,255]))
        health = min(np.sum(green>0)/green.size*3, 1.0)

        gray = cv2.cvtColor(ammo_zone, cv2.COLOR_RGB2GRAY)
        ammo = min(np.sum(gray>200)/gray.size*3, 1.0)

        hsv = cv2.cvtColor(enemy_zone, cv2.COLOR_RGB2HSV)
        r1 = cv2.inRange(hsv, np.array([0,50,50]), np.array([10,255,255]))
        r2 = cv2.inRange(hsv, np.array([160,50,50]), np.array([180,255,255]))
        enemy = (np.sum(cv2.bitwise_or(r1,r2)>0) / hsv.size) > 0.08

        return {"health": float(health), "ammo": float(ammo), "enemy": bool(enemy)}

    def process(self, image_b64, client_state=None):
        self.frame_count += 1
        img = self.decode_image(image_b64)
        state = self.analyze_frame(img)

        # Heurísticas
        if state["health"] < 0.25:
            return {"action": "HEAL", "x": 150, "y": 800, "duration": 500, "state": state}
        if state["enemy"] and state["ammo"] < 0.1:
            return {"action": "RELOAD", "x": 1200, "y": 900, "duration": 1500, "state": state}
        if state["enemy"]:
            return {"action": "SHOOT", "x": 1150, "y": 750, "duration": 100, "state": state}

        # IA
        vec = torch.FloatTensor([state["health"], state["ammo"], float(state["enemy"]), 1.0, 0.5]).to(self.device)
        with torch.no_grad():
            probs = self.model(torch.cat([vec, torch.zeros(123).to(self.device)]))
            action_idx = torch.multinomial(probs, 1).item()

        name, x, y, dur = self.ACTIONS.get(action_idx, ("HOLD", 960, 700, 100))
        return {"action": name, "x": x, "y": y, "duration": dur, "state": state}
'''

with open('/content/ffai_core/engine.py', 'w') as f:
    f.write(engine_code)

print("✅ engine.py creado")

✅ engine.py creado


## Celda 4: Configurar GPU

In [12]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"🚀 GPU Tesla T4 activa")
    print(f"🔥 Memoria: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("⚠️ Usando CPU (más lento)")
    print("💡 Runtime → Change runtime type → GPU")

🚀 GPU Tesla T4 activa
🔥 Memoria: 15.6 GB


## Celda 5: Inicializar Motor IA

In [13]:
from ffai_core.engine import FFAIEngine

# Crear motor
engine = FFAIEngine(device=device)

# Intentar cargar modelo desde Drive si existe
import os
model_path = "/content/drive/MyDrive/FFAI/models/ff_model.pt"
if os.path.exists(model_path):
    print(f"📂 Cargando modelo desde Drive...")
    # engine.load_model(model_path)  # Implementar si hay modelos guardados
else:
    print(f"💾 El modelo se guardará en: {model_path}")
    os.makedirs(os.path.dirname(model_path), exist_ok=True)

print("✅ Motor IA listo")

🧠 Engine listo en cuda
💾 El modelo se guardará en: /content/drive/MyDrive/FFAI/models/ff_model.pt
✅ Motor IA listo


## Celda 6: Crear API Flask

In [14]:
from flask import Flask, request, jsonify
from flask_socketio import SocketIO, emit
from flask_cors import CORS
import json
import time

app = Flask(__name__)
CORS(app)
socketio = SocketIO(app, cors_allowed_origins="*", async_mode="threading")

# Stats
stats = {
    "started": time.time(),
    "frames_processed": 0,
    "clients_connected": 0
}

# Rutas HTTP
@app.route("/")
def root():
    return jsonify({
        "status": "🟢 Online",
        "service": "FFAI API",
        "version": "3.1.0",
        "device": device,
        "stats": {
            "frames": stats["frames_processed"],
            "uptime": int(time.time() - stats["started"])
        }
    })

@app.route("/health")
def health():
    return jsonify({"status": "ok"})

# WebSocket
@socketio.on("connect")
def handle_connect():
    stats["clients_connected"] += 1
    print(f"📱 Cliente conectado. Total: {stats['clients_connected']}")
    emit("connected", {"status": "ok"})

@socketio.on("disconnect")
def handle_disconnect():
    stats["clients_connected"] = max(0, stats["clients_connected"] - 1)
    print(f"📴 Cliente desconectado. Total: {stats['clients_connected']}")

@socketio.on("frame")
def handle_frame(data):
    try:
        if isinstance(data, str):
            data = json.loads(data)

        # Procesar con IA
        result = engine.process(
            data.get("imageBase64", ""),
            data.get("clientState", {})
        )

        stats["frames_processed"] += 1

        # Enviar respuesta
        emit("action", {
            "type": "action",
            "action": result["action"],
            "coordinates": {"x": result["x"], "y": result["y"]},
            "duration": result["duration"],
            "game_state": result["state"]
        })

    except Exception as e:
        print(f"Error: {e}")
        emit("error", {"message": str(e)})

print("✅ API Flask configurada")

✅ API Flask configurada


## Celda 7: Conectar Ngrok (URL Pública)

In [15]:
from pyngrok import ngrok
import nest_asyncio

# Patch para asyncio
nest_asyncio.apply()

# Token ngrok configurado (URL permanente)
NGROK_TOKEN = "3CU4gdr4ZZBNCaiS2SS0M65EZZH_55e54r4qkAkq8aEu2MQE8"
ngrok.set_auth_token(NGROK_TOKEN)

# Crear túnel
public_url = ngrok.connect(5000, "http")
print("\n" + "="*60)
print("🌐 URL PÚBLICA DEL SERVIDOR:")
print(f"   {public_url}")
print("="*60)
print(f"\n📋 Configura tu APK:")
print(f'   SERVER_WS_URL = "wss://{public_url.public_url.replace("https://", "").replace("http://", "")}/ws"')
print(f"\n✅ Token ngrok configurado - URL estable")


🌐 URL PÚBLICA DEL SERVIDOR:
   NgrokTunnel: "https://poem-tipping-glitter.ngrok-free.dev" -> "http://localhost:5000"

📋 Configura tu APK:
   SERVER_WS_URL = "wss://poem-tipping-glitter.ngrok-free.dev/ws"

✅ Token ngrok configurado - URL estable


## Celda 8: 🚀 INICIAR SERVIDOR

In [ ]:
print("🚀 Iniciando servidor FFAI...")
print("   Presiona STOP cuando quieras detener\n")

# Iniciar
socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

🚀 Iniciando servidor FFAI...
   Presiona STOP cuando quieras detener

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


## Bonus: Guardar Checkpoint

Ejecuta esto para guardar el modelo en Drive:

In [ ]:
# Guardar checkpoint manualmente
checkpoint_path = "/content/drive/MyDrive/FFAI/checkpoints/manual_save.pth"
torch.save({
    'model_state_dict': engine.model.state_dict(),
    'frame_count': engine.frame_count,
    'timestamp': time.time()
}, checkpoint_path)
print(f"✅ Checkpoint guardado: {checkpoint_path}")